# 02 — LDA em Reuters-21578

## O que esse notebook faz

Aplica **Latent Dirichlet Allocation (LDA, Blei et al. 2003)** ao corpus Reuters-21578 (notícias econômicas em inglês, 1987) para descobrir tópicos latentes via inferência probabilística (Variational Bayes, gensim `LdaMulticore`).

## O que é LDA, intuitivamente

LDA assume que:
1. Cada documento é uma **mistura de tópicos** (distribuição θ sobre tópicos)
2. Cada tópico é uma **distribuição de palavras** (distribuição φ sobre vocabulário)
3. As palavras de um documento são geradas amostrando: primeiro um tópico de θ, depois uma palavra de φ

O algoritmo "inverte" esse processo generativo: dado os documentos, recupera θ e φ que melhor explicam os dados. As distribuições φ são os "tópicos" — listas de palavras com pesos diferentes.

## Por que Reuters-21578 como corpus de validação

- **Benchmark canônico**: usado desde Blei (2003), Apté & Damerau (1994); ground truth implícito (categorias do split ModApte).
- **Domínio jornalístico** alinhado ao framework do projeto.
- **Volume razoável** (~10.5k docs após limpeza) — viável para grid search + multi-seed.
- **Idioma EN** — testa o pipeline `lemmatize.py` na trilha inglês antes de Folha (PT-BR).

## Métricas que veremos

| Métrica | O que mede | Faixa típica |
|---|---|---|
| **c_v coherence** (Roder et al. 2015) | Coerência semântica das top palavras dos tópicos via co-ocorrência de janela | 0.3-0.7 (>0.5 bom) |
| **Exclusivity** | Quanto as keywords são distintivas de UM tópico | 0.5-1.0 |
| **Topic Diversity (Dieng et al. 2020)** | Razão de palavras únicas no top-K consolidado | 0.7-0.9 ideal |
| **Diversity entropy** | Entropia normalizada das distribuições θ por doc | ~0.95 = bem espalhado |

## Output esperado

`data/output/reuters/lda_results.csv`, `lda_topics_for_eval.csv`, `lda_metrics.csv`.

## 1. Setup e imports

Carregamos parâmetros do `configs/params.yaml` (entrada `corpora.reuters`), o seed global e as funções dos pipelines reusáveis:

- `_helpers.lemmatize_corpus(docs, lang, params)` → tokens lematizados + Dictionary com filter_extremes
- `_helpers.grid_search_k / train_lda / extract_topics_keywords / compute_doc_distributions` → fit e extração
- `_helpers.name_all_topics` → rotulagem via Ollama (modelo em `params.yaml`)
- `_helpers.*` → métricas (c_v, exclusivity, FREX, NPMI, etc.)

In [ ]:
# ── path fix: resolve _helpers.py e data/ de qualquer subdiretório ──
import sys as _sys, os as _os
from pathlib import Path as _P
_here = _P().resolve()
_nb_dir = _here if (_here / '_helpers.py').exists() else _here.parent
if str(_nb_dir) not in _sys.path:
    _sys.path.insert(0, str(_nb_dir))
_os.chdir(_nb_dir)          # '../data/...' paths ficam corretos
del _here, _nb_dir, _P, _sys, _os
# ─────────────────────────────────────────────────────────────────────
# Cell 1 — Imports
import sys, os, json, time, ast
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from wordcloud import WordCloud
from collections import Counter
from gensim.models import LdaMulticore

from _helpers import load_params, get_corpus_config, get_seed, load_corpus
from _helpers import lemmatize_corpus
from _helpers import (
    grid_search_k, grid_search_alpha_eta, train_lda, extract_topics_keywords,
    compute_doc_distributions, qualitative_report,
)
from _helpers import name_all_topics
from _helpers import prompt_builder_en_news, system_prompt_en_news
from _helpers import (
    compute_coherence_cv, compute_exclusivity, compute_diversity,
    compute_topic_diversity, compute_stability, export_results, export_topics_for_eval,
)

sns.set_style("whitegrid")
plt.rcParams["figure.dpi"] = 100


In [ ]:
# Cell 2 — Carrega params do params.yaml
CORPUS_ID = "reuters"
LANG = "en"

params = load_params()
_, cfg = get_corpus_config(params, CORPUS_ID)
seed = get_seed(params)
TEXT_COL = cfg["text_column"]

print(f"corpus_id = {CORPUS_ID}")
print(f"text_column = {TEXT_COL}")
print(f"language = {LANG}")
print(f"seed = {seed}")
print(f"covariates (usadas no STM, não no LDA) = {cfg.get('covariates', [])}")

## 2. Corpus Reuters-21578

### Origem e schema

- Carregado via `nltk.corpus.reuters` (split ModApte, 10.788 fileids)
- Adapter: `01-preprocessing/src/adapters/reuters.py`
- Filtros aplicados: `len(message.split()) >= 20` (descarta docs muito curtos)
- Schema padronizado: `post_id, message, date, category, lang`

### Limitações

- **Date constante**: `1987-01-01` para todos (nltk não expõe granularidade temporal). STM perde poder de `s(date)`.
- **Multi-label collapse**: Reuters tem múltiplas categorias por doc; mantemos só a primeira (mais frequente).

In [ ]:
# Cell 3 — Carrega corpus_limpo.csv
_corpus_dir = f"../data/input/{CORPUS_ID}/"
assert os.path.exists(_corpus_dir), (
    f"NÃO ENCONTRADO: {_corpus_dir}\n"
    f"Copie corpus_limpo.csv de 01-preprocessing/data/output/{CORPUS_ID}/ para este diretório."
)
df = load_corpus(_corpus_dir)
docs = df[TEXT_COL].astype(str).tolist()
post_ids = df["post_id"].astype(str).tolist()

print(f"Total docs: {len(docs):,}")
print(f"Colunas: {list(df.columns)}")
df.head(3)


In [ ]:
# Cell 4 — Estatisticas descritivas + distribuicao de tokens
df["n_tokens"] = df["message"].str.split().str.len()

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].hist(df["n_tokens"], bins=60, edgecolor="white", color="steelblue")
axes[0].axvline(df["n_tokens"].median(), color="red", linestyle="--", label=f'mediana={df["n_tokens"].median():.0f}')
axes[0].axvline(df["n_tokens"].mean(), color="orange", linestyle="--", label=f'média={df["n_tokens"].mean():.0f}')
axes[0].set_xlabel("Tokens por documento"); axes[0].set_ylabel("# docs")
axes[0].set_title("Distribuição de tamanho dos documentos"); axes[0].legend()
axes[0].set_xlim(0, df["n_tokens"].quantile(0.99))

cat_counts = df["category"].value_counts().head(15)
axes[1].barh(cat_counts.index[::-1], cat_counts.values[::-1], color="steelblue")
axes[1].set_xlabel("# documentos"); axes[1].set_title("Top 15 categorias (primary label)")
plt.tight_layout(); plt.show()

print(f"\nDescritivas n_tokens: {df['n_tokens'].describe().to_dict()}")

## 3. Lematização e construção do vocabulário

### O que `_helpers.lemmatize_corpus` faz

1. Carrega spaCy `en_core_web_sm` (cached em module-level)
2. Para cada doc: tokeniza, descarta stopwords (lista interna spaCy) + tokens não-alfa + tokens curtos (`len(lemma) <= 2`)
3. Constrói gensim `Dictionary` com `filter_extremes(no_below=5, no_above=0.5)`:
   - Remove palavras que aparecem em <5 docs (raras, ruído)
   - Remove palavras em >50% dos docs (genéricas, sem poder discriminante — ex.: "said")

### Por que lematizar (vs Porter stemming)

- Lematização preserva forma legível (`running` → `run`; STM via R usa Porter stemming agressivo: `running` → `run`, `companies` → `compani`)
- Mais interpretável para a banca da dissertação
- Custo: ~30s a ~3min em CPU dependendo do volume

In [ ]:
# Cell 5 — Lematiza corpus
print("Lematizando (single-process spaCy en_core_web_sm)...")
t0 = time.time()
tokenized, dictionary = lemmatize_corpus(docs, LANG, params)
print(f"  done em {time.time()-t0:.0f}s")
print(f"  vocab final: {len(dictionary):,} palavras (após filter_extremes)")

# Estatísticas dos tokens lemmatizados
n_tokens_lemma = [len(t) for t in tokenized]
n_unique = [len(set(t)) for t in tokenized]

print(f"\n  avg tokens/doc após lemmatize: {np.mean(n_tokens_lemma):.1f}")
print(f"  avg unique tokens/doc: {np.mean(n_unique):.1f}")

In [ ]:
# Cell 6 — Top 30 palavras mais frequentes no vocabulário final
all_tokens = [t for doc in tokenized for t in doc]
freq = Counter(all_tokens)
top30 = freq.most_common(30)

fig, ax = plt.subplots(figsize=(12, 6))
words, counts = zip(*top30)
ax.barh(words[::-1], counts[::-1], color="steelblue")
ax.set_xlabel("Frequência total no corpus")
ax.set_title("Top 30 palavras (após lemmatize + filter_extremes)")
plt.tight_layout(); plt.show()

In [ ]:
# Cell 7 — Bag-of-Words representation
corpus_bow = [dictionary.doc2bow(d) for d in tokenized]
print(f"corpus_bow: {len(corpus_bow):,} docs")
print(f"avg unique tokens/doc no BoW: {np.mean([len(b) for b in corpus_bow]):.1f}")
print(f"\nExemplo doc[0] BoW (primeiros 10 pares (word_id, count)):")
for word_id, count in corpus_bow[0][:10]:
    print(f"  {dictionary[word_id]:20s}  count={count}")

## 4. Seleção de K (número de tópicos)

### Por que K importa

K é o **único hiperparâmetro estrutural** do LDA. Influencia a granularidade dos tópicos:
- K muito baixo → tópicos macro/genéricos (varios temas misturados em um)
- K muito alto → fragmentação (mesmo tema dividido em múltiplos clusters)

### Como escolher

Padrão na literatura: rodar LDA em vários K, calcular **c_v coherence** (Roder et al. 2015) — métrica que combina:
- co-ocorrência das top-N palavras numa janela deslizante
- score NPMI normalizado
- agregação via cosine similarity

Limitação conhecida: **c_v penaliza K alto** (Stevens et al. 2012). Por isso complementamos com inspeção qualitativa em múltiplos K (próxima célula).

In [ ]:
# Cell 8 — Grid search K (carrega resultado do run standalone se existir)
from _helpers import make_run_output_dir
BASE_OUTPUT_DIR = Path(f"../data/output/{CORPUS_ID}")
BASE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
# Saida carimbada por execucao: <corpus>_<data>_<hora> (nao sobrescreve runs anteriores)
OUTPUT_DIR = make_run_output_dir(BASE_OUTPUT_DIR, CORPUS_ID)
out_dir = str(OUTPUT_DIR)
# Cache do grid search: le/persiste no diretorio-BASE (reutiliza entre runs)
m_path = f"{BASE_OUTPUT_DIR}/lda_metrics.csv"

if os.path.exists(m_path):
    m = pd.read_csv(m_path)
    scores = ast.literal_eval(m.iloc[0]["k_grid_scores"])
    scores = {int(k): float(v) for k, v in scores.items()}
    _pkey = "k_perplexity_scores"
    if _pkey in m.columns and pd.notna(m.iloc[0].get(_pkey)):
        perplexity_scores = ast.literal_eval(str(m.iloc[0][_pkey]))
        perplexity_scores = {int(k): float(v) for k, v in perplexity_scores.items()}
    else:
        perplexity_scores = {}
    print(f"Carregado grid pre-computado de {m_path}")
    print(f"  K range: {sorted(scores.keys())}")
    print(f"  passes: {m.iloc[0].get('grid_passes', '?')}")
else:
    K_RANGE = [3, 5, 7, 8, 10, 12, 15, 20, 25, 30]
    print(f"Rodando grid K em {K_RANGE} passes=20 (~13min)...")
    scores, perplexity_scores = grid_search_k(
        corpus_bow, dictionary, tokenized, K_RANGE, seed=seed, passes=20)
    print(f"Done.")

scores

In [ ]:
# Cell 9 — Visualizacao c_v vs K
fig, ax = plt.subplots(figsize=(11, 5))
ks = sorted(scores)
vs = [scores[k] for k in ks]

ax.plot(ks, vs, marker="o", linewidth=2.5, markersize=9, color="steelblue", label="C_v (eixo esq.)")
ax.fill_between(ks, vs, alpha=0.1, color="steelblue")

peak_k = max(scores, key=scores.get)
ax.axvline(peak_k, color="red", linestyle="--", alpha=0.6, label=f"peak c_v: K={peak_k} ({scores[peak_k]:.3f})")
ax.axvline(10, color="green", linestyle=":", alpha=0.7, linewidth=2, label="K=10 (top-10 ModApte canonical)")

# Annotate each K
for k, v in zip(ks, vs):
    delta = (v - scores[peak_k]) / scores[peak_k] * 100
    label = f"{v:.3f}" if k == peak_k else f"{v:.3f}\n({delta:+.0f}%)"
    ax.annotate(label, (k, v), textcoords="offset points", xytext=(0, 10),
                ha="center", fontsize=8)

ax.set_xlabel("K (número de tópicos)"); ax.set_ylabel("c_v coherence", color="steelblue")
ax.set_title(f"LDA grid search — Reuters-21578 ({len(docs):,} docs)")
if perplexity_scores:
    ax2 = ax.twinx()
    pks = sorted(perplexity_scores)
    pvs = [perplexity_scores[k] for k in pks]
    ax2.plot(pks, pvs, marker="s", linewidth=2, markersize=7, color="darkorange",
             linestyle="--", label="Perplexidade (eixo dir.)")
    ax2.set_ylabel("Perplexidade (menor = melhor fit)", color="darkorange")
    ax2.tick_params(axis="y", colors="darkorange")
    lines2, labels2 = ax2.get_legend_handles_labels()
    lines1, labels1 = ax.get_legend_handles_labels()
    ax.legend(lines1 + lines2, labels1 + labels2, loc="lower right")
else:
    ax.legend(loc="lower right")
plt.tight_layout(); plt.show()

print(f"\nPeak: K={peak_k} (c_v={scores[peak_k]:.4f})")
if 10 in scores:
    print(f"K=10: c_v={scores[10]:.4f} (delta vs peak: {(scores[10]-scores[peak_k])/scores[peak_k]*100:+.1f}%)")

## 5. Inspeção qualitativa multi-K

Como `c_v` favorece K menor, precisamos olhar **os tópicos em si** para escolher K com critério metodológico (não só estatístico). Carregamos os tópicos de K=5, 7, 10, 15, 30 pré-computados em `lda_multi_k_inspect.csv`, se disponível (execução prévia opcional — ver célula abaixo).

In [ ]:
# Cell 10 — Multi-K topics side by side (opcional, requer lda_multi_k_inspect.csv pre-computado)
multi_path = f"{out_dir}/lda_multi_k_inspect.csv"
if os.path.exists(multi_path):
    multi = pd.read_csv(multi_path)
    for k_val in sorted(multi["k"].unique()):
        sub = multi[multi["k"] == k_val].sort_values("topic_id")
        print(f"=== K={k_val} ===")
        for _, row in sub.iterrows():
            kws = row["keywords"].split(", ")[:8]
            print(f"  T{int(row['topic_id']):2d}: {', '.join(kws)}")
        print()
else:
    print("lda_multi_k_inspect.csv nao encontrado — pulando inspecao multi-K (opcional).")

In [ ]:
# Cell 11 — Cobertura ModApte top-10 por K (analise visual)
modapte_top10 = ["earn", "acq", "money-fx", "grain", "crude", "trade", "interest", "ship", "wheat", "corn"]

# Mapeamento manual (baseado em inspecao visual dos topicos por K, ver Cell 10 acima)
coverage = {
    5:  {"earn": 1, "acq": 1, "money-fx": 0.5, "grain": 0.3, "crude": 0.3, "trade": 1, "interest": 0.5, "ship": 0, "wheat": 0, "corn": 0},
    10: {"earn": 1, "acq": 1, "money-fx": 1, "grain": 1, "crude": 1, "trade": 1, "interest": 1, "ship": 0, "wheat": 1, "corn": 1},
    15: {"earn": 1, "acq": 1, "money-fx": 1, "grain": 1, "crude": 1, "trade": 1, "interest": 1, "ship": 0, "wheat": 1, "corn": 1},
    30: {"earn": 1, "acq": 1, "money-fx": 1, "grain": 1, "crude": 1, "trade": 1, "interest": 1, "ship": 0.5, "wheat": 1, "corn": 1},
}
cov_df = pd.DataFrame(coverage).T
cov_df = cov_df[modapte_top10]

fig, ax = plt.subplots(figsize=(12, 4))
sns.heatmap(cov_df, annot=True, cmap="RdYlGn", vmin=0, vmax=1,
            cbar_kws={"label": "Cobertura (1=pleno, 0.5=parcial, 0=ausente)"},
            linewidths=0.5, ax=ax)
ax.set_xlabel("Categoria ModApte canonical"); ax.set_ylabel("K (núm. tópicos)")
ax.set_title("Cobertura das top-10 categorias do split ModApte por K (Apté & Damerau 1994)")
plt.tight_layout(); plt.show()

cov_df["score"] = cov_df.sum(axis=1)
print("\nScore total (max=10):"); print(cov_df["score"])

## 6. Decisão de K e treino final

**Trade-off resumido:**

| K | c_v | ModApte coverage | Comentário |
|---|---|---|---|
| 5 | 0.529 (peak) | 4/10 (T1 mistura) | macro robusto, defesa: pico estatístico |
| 10 | 0.475 (-10%) | 8/10 (9/10 coesos) | sweet spot |
| 15 | 0.479 (-10%) | 9/10 + dividends | granular mas T5 piora |
| 30 | 0.483 (-9%) | 10/10 + temas extras | overfit |

Edite `BEST_K` na próxima célula. **Padrão recomendado: K=10**.

In [ ]:
# Cell 12 — Grid alpha/eta (K=BEST_K fixo) + treino final
BEST_K = max(scores, key=scores.get)  # peak C_v — revise o gráfico e ajuste se necessário
print(f"BEST_K automático (peak C_v): {BEST_K}")

def _parse_ae_row(row):
    """Restaura tipos Python a partir de linha do CSV de cache."""
    alpha = str(row["alpha"])
    eta = row["eta"]
    if alpha not in ("symmetric", "asymmetric"):
        try: alpha = float(alpha)
        except: alpha = "symmetric"
    if pd.isna(eta) or str(eta) in ("None", "nan", ""):
        eta = None
    elif str(eta) != "auto":
        try: eta = float(eta)
        except: eta = None
    return alpha, eta

_ae_cache = f"{BASE_OUTPUT_DIR}/lda_alpha_eta_grid.csv"
_alpha_grid = params["evaluation"].get("lda_alpha_grid", ["symmetric", "asymmetric", 0.1, 0.5])
_eta_grid   = params["evaluation"].get("lda_eta_grid", [None, 0.01, 0.1])

if os.path.exists(_ae_cache):
    ae_df = pd.read_csv(_ae_cache)
    best_alpha, best_eta = _parse_ae_row(ae_df.iloc[0])
    print(f"Grid alpha/eta carregado do cache ({_ae_cache}).")
else:
    print(f"Grid alpha ({len(_alpha_grid)}) × eta ({len(_eta_grid)}) = {len(_alpha_grid)*len(_eta_grid)} combos, passes=10...")
    t0 = time.time()
    ae_df, best_alpha, best_eta = grid_search_alpha_eta(
        corpus_bow, dictionary, tokenized, BEST_K,
        alpha_grid=_alpha_grid, eta_grid=_eta_grid, seed=seed, passes=10,
    )
    ae_df.to_csv(_ae_cache, index=False)
    print(f"  done em {time.time()-t0:.0f}s")

print(ae_df.to_string(index=False))
print(f"\n  ★ best_alpha={best_alpha!r}  best_eta={best_eta!r}")

# Heatmap C_v por alpha × eta
if len(ae_df) > 1:
    _pivot = ae_df.pivot_table(index="alpha", columns="eta", values="cv", aggfunc="mean")
    fig, ax = plt.subplots(figsize=(9, max(3, len(_pivot) * 0.8 + 1)))
    sns.heatmap(_pivot, annot=True, fmt=".4f", cmap="RdYlGn", ax=ax, linewidths=0.5)
    ax.set_title(f"LDA alpha × eta grid (K={BEST_K}) — C_v coherence (maior = melhor)")
    plt.tight_layout(); plt.show()

# Treino final com melhores hiperparâmetros
print(f"\nTreinando LDA K={BEST_K}, alpha={best_alpha!r}, eta={best_eta!r}, passes=20...")
t0 = time.time()
model = train_lda(corpus_bow, dictionary, k=BEST_K, seed=seed, passes=20, alpha=best_alpha, eta=best_eta)
print(f"  done em {time.time()-t0:.0f}s")

top_n = params["evaluation"]["top_n_keywords"]
topics_keywords = extract_topics_keywords(model, k=BEST_K, top_n=top_n)

print(f"\nTopics (top-{top_n}):")
for tid in sorted(topics_keywords):
    print(f"  T{tid:2d}: {topics_keywords[tid]}")

## 7. Visualização dos tópicos

Wordclouds + bar charts mostram a estrutura semântica de cada tópico. Escala logarítmica destaca palavras menos frequentes mas igualmente representativas.

In [ ]:
# Cell 13 — Wordclouds (1 por topico)
ncols = 3
nrows = (BEST_K + ncols - 1) // ncols
fig, axes = plt.subplots(nrows, ncols, figsize=(15, 4 * nrows))
axes = axes.flatten() if BEST_K > 1 else [axes]

for tid in sorted(topics_keywords):
    # gensim show_topic retorna (palavra, prob); usar prob como freq do wordcloud
    word_probs = dict(model.show_topic(tid, topn=30))
    wc = WordCloud(width=600, height=400, background_color="white",
                   colormap="viridis", max_words=30, prefer_horizontal=0.9).generate_from_frequencies(word_probs)
    axes[tid].imshow(wc, interpolation="bilinear")
    axes[tid].set_title(f"T{tid}", fontsize=12, fontweight="bold")
    axes[tid].axis("off")

# Esconde axes nao usados
for i in range(BEST_K, len(axes)):
    axes[i].axis("off")

plt.tight_layout(); plt.show()

In [ ]:
# Cell 14 — Bar chart top-10 keywords per topic
fig, axes = plt.subplots(nrows, ncols, figsize=(15, 3.2 * nrows))
axes = axes.flatten() if BEST_K > 1 else [axes]

for tid in sorted(topics_keywords):
    word_probs = model.show_topic(tid, topn=10)
    words, probs = zip(*word_probs)
    axes[tid].barh(list(words)[::-1], list(probs)[::-1], color="steelblue")
    axes[tid].set_title(f"T{tid}", fontsize=11, fontweight="bold")
    axes[tid].set_xlabel("P(palavra | tópico)")
    axes[tid].tick_params(labelsize=9)

for i in range(BEST_K, len(axes)):
    axes[i].axis("off")

plt.tight_layout(); plt.show()

In [ ]:
import seaborn as sns
# Heatmap phi: top palavras x topicos
_top_n_phi = 25
_all_words = []
for tid in range(BEST_K):
    for w, _ in model.show_topic(tid, topn=_top_n_phi):
        if w not in _all_words:
            _all_words.append(w)
_all_words = _all_words[:min(50, len(_all_words))]

_phi = np.zeros((len(_all_words), BEST_K))
for tid in range(BEST_K):
    _td = dict(model.show_topic(tid, topn=200))
    for wi, w in enumerate(_all_words):
        _phi[wi, tid] = _td.get(w, 0.0)

import textwrap as _tw_lda
_col_labels = [f"T{t}:\n{_tw_lda.shorten(str(names.get(t,"")), 18, placeholder="...")}" for t in range(BEST_K)]
_phi_df = pd.DataFrame(_phi, index=_all_words, columns=_col_labels)

fig, ax = plt.subplots(figsize=(max(10, BEST_K * 0.8), max(8, len(_all_words) * 0.32)))
sns.heatmap(_phi_df, cmap="YlOrRd", ax=ax, linewidths=0.15,
            cbar_kws={"label": "p(word|topic) phi"})
ax.set_title("Distribuicao phi — palavras x topicos (LDA)", fontsize=12, fontweight="bold")
ax.set_xlabel("Topico"); ax.set_ylabel("Palavra")
plt.xticks(fontsize=8); plt.yticks(fontsize=8)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "lda_heatmap_phi.png", dpi=150, bbox_inches="tight", facecolor="white")
plt.show()
print(f"Salvo: {OUTPUT_DIR / 'lda_heatmap_phi.png'}")


In [ ]:
import seaborn as sns
# Similaridade cosseno entre topicos (vetores phi)
from sklearn.metrics.pairwise import cosine_similarity

_phi_topics = np.zeros((BEST_K, len(dictionary)))
for tid in range(BEST_K):
    for word_id, prob in model.get_topic_terms(tid, topn=len(dictionary)):
        _phi_topics[tid, word_id] = prob

_cos_sim = cosine_similarity(_phi_topics)
import textwrap as _tw_cos
_tlabels = [f"T{t}: {_tw_cos.shorten(str(names.get(t,"")), 20, placeholder="...")}" for t in range(BEST_K)]

fig, ax = plt.subplots(figsize=(max(8, BEST_K * 0.65), max(7, BEST_K * 0.6)))
sns.heatmap(_cos_sim, xticklabels=_tlabels, yticklabels=_tlabels,
            cmap="RdYlGn_r", vmin=0, vmax=1,
            annot=(BEST_K <= 20), fmt=".2f", ax=ax,
            annot_kws={"fontsize": 7},
            cbar_kws={"label": "Cosine similarity (phi)"})
ax.set_title("Similaridade entre topicos (cosine phi) — LDA", fontsize=12, fontweight="bold")
plt.xticks(rotation=45, ha="right", fontsize=8)
plt.yticks(fontsize=8)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "lda_heatmap_topicos_cosine.png", dpi=150, bbox_inches="tight", facecolor="white")
plt.show()
print(f"Salvo: {OUTPUT_DIR / 'lda_heatmap_topicos_cosine.png'}")


## 8. Rotulagem via LLM (Ollama)

`_helpers.name_all_topics` envia as keywords pro modelo configurado em `params.yaml` (`llm.model`) via Ollama com prompt few-shot. Em ~9s/tópico em CPU. Fallback: top-3 keywords se Ollama indisponível.

Corpus EN (`LANG="en"`): usa `lang="en"` + `prompt_builder_en_news`/`system_prompt_en_news` (few-shot em ingles, dominio noticias/financeiro) para gerar rotulos em ingles em vez de PT-BR.

In [ ]:
# Cell 15 — Naming
llm_cfg = params.get("llm", {})
print(f"Naming via {llm_cfg.get('model')} ({BEST_K} tópicos × ~9s = ~{BEST_K*9}s) [lang={LANG}]...")

t0 = time.time()
try:
    names = name_all_topics(
        topics_keywords,
        model=llm_cfg["model"],
        base_url=llm_cfg.get("base_url", "http://localhost:11434"),
        lang=LANG,
        prompt_builder=prompt_builder_en_news if LANG == "en" else None,
        system_prompt=system_prompt_en_news if LANG == "en" else None,
    )
    print(f"  done em {time.time()-t0:.0f}s")
except Exception as e:
    print(f"  LLM falhou ({e}) — fallback top-3")
    names = {tid: ", ".join(kws[:3]) for tid, kws in topics_keywords.items()}

print()
for tid in sorted(names):
    print(f"  T{tid:2d}: {names[tid]}")
    print(f"        keywords: {topics_keywords[tid][:8]}")

## 9. Métricas quantitativas

- **c_v** recomputada (validação cruzada do grid)
- **Exclusivity**: 1 - prop. de keywords compartilhadas entre tópicos
- **Topic Diversity**: razão de palavras únicas no consolidado dos top-K
- **Diversity entropy** das distribuições θ doc-tópico (quão espalhadas)

In [ ]:
# Cell 16 — Calcula metricas
print("Calculando métricas...")
dominant, full = compute_doc_distributions(model, corpus_bow, k=BEST_K)

metrics = {
    "model": "lda",
    "topic_type": "probabilistic",
    "granularity": "unit",
    "corpus": CORPUS_ID,
    "n_topics": BEST_K,
    "best_k_cv": float(scores.get(BEST_K, 0.0)),
    "exclusivity": float(compute_exclusivity(topics_keywords)),
    "diversity_entropy": float(compute_diversity(full)),
    "topic_diversity_dieng": float(compute_topic_diversity(topics_keywords)),
}
metrics["coherence_cv_recomputed"] = float(compute_coherence_cv(topics_keywords, tokenized, dictionary))

# Visualiza num radar
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bar chart
metric_keys = ["best_k_cv", "coherence_cv_recomputed", "exclusivity",
               "topic_diversity_dieng", "diversity_entropy"]
metric_vals = [metrics[k] for k in metric_keys]
colors = ["#1f77b4", "#ff7f0e", "#2ca02c", "#d62728", "#9467bd"]
axes[0].bar(metric_keys, metric_vals, color=colors)
axes[0].set_ylim(0, 1.05); axes[0].set_ylabel("Score (0-1)")
axes[0].set_title(f"Metricas LDA K={BEST_K}")
axes[0].tick_params(axis="x", rotation=20, labelsize=9)
for i, (k, v) in enumerate(zip(metric_keys, metric_vals)):
    axes[0].text(i, v + 0.02, f"{v:.3f}", ha="center", fontsize=10, fontweight="bold")

# Radar
angles = np.linspace(0, 2*np.pi, len(metric_keys), endpoint=False).tolist()
vals = metric_vals + [metric_vals[0]]
angles_p = angles + [angles[0]]
axes[1] = plt.subplot(122, projection="polar")
axes[1].plot(angles_p, vals, "o-", linewidth=2, color="steelblue")
axes[1].fill(angles_p, vals, alpha=0.25, color="steelblue")
axes[1].set_xticks(angles); axes[1].set_xticklabels(metric_keys, fontsize=8)
axes[1].set_ylim(0, 1)
axes[1].set_title("Visao radar")

plt.tight_layout(); plt.show()
print(json.dumps(metrics, indent=2))

In [ ]:
# Cell 17 — c_v por topico (nao so media)
from gensim.models import CoherenceModel
cm = CoherenceModel(
    topics=[topics_keywords[tid] for tid in sorted(topics_keywords)],
    texts=tokenized, dictionary=dictionary, coherence="c_v",
)
per_topic_cv = cm.get_coherence_per_topic()

fig, ax = plt.subplots(figsize=(11, 4))
tids = sorted(topics_keywords)
labels = [f"T{tid}\n{names[tid][:22]}" for tid in tids]
bars = ax.bar(tids, per_topic_cv, color=["#2ca02c" if c >= 0.5 else "#ff7f0e" if c >= 0.4 else "#d62728" for c in per_topic_cv])
ax.axhline(np.mean(per_topic_cv), color="black", linestyle="--", alpha=0.5, label=f"média={np.mean(per_topic_cv):.3f}")
ax.set_xticks(tids); ax.set_xticklabels(labels, fontsize=8, rotation=30, ha="right")
ax.set_ylabel("c_v coherence"); ax.set_title("Coerência c_v por tópico (verde >=0.5, laranja 0.4-0.5, vermelho <0.4)")
ax.legend()
for bar, c in zip(bars, per_topic_cv):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01, f"{c:.3f}",
            ha="center", fontsize=8)
plt.tight_layout(); plt.show()

In [13]:
stability_seeds = params["evaluation"]["stability_seeds"]
results_per_seed = {}

for s in stability_seeds:
    print(f"Executando seed={s}...")
    model_s = LdaMulticore(
        corpus=corpus_bow,
        id2word=dictionary,
        num_topics=BEST_K,
        random_state=s,
        passes=20,
        workers=2,
    )
    seed_keywords = {}
    for tid in range(BEST_K):
        words = model_s.show_topic(tid, topn=top_n)
        seed_keywords[tid] = [w for w, _ in words]
    results_per_seed[s] = seed_keywords

    seed_topics = []
    for bow in corpus_bow:
        dist = model_s.get_document_topics(bow, minimum_probability=0.0)
        seed_topics.append(max(dist, key=lambda x: x[1])[0])
    seed_df = pd.DataFrame({"doc_id": range(len(seed_topics)), "topic_id": seed_topics})
    seed_df.to_csv(OUTPUT_DIR / f"lda_topics_seed{s}.csv", index=False)

mean_j, std_j = compute_stability(results_per_seed)
print("\nEstabilidade LDA:")
print(f"  Jaccard médio: {mean_j:.4f}")
print(f"  Desvio padrão: {std_j:.4f}")


Executando seed=42...


Executando seed=123...


Executando seed=456...


Executando seed=789...


Executando seed=1024...



Estabilidade LDA:
  Jaccard médio: 0.3284
  Desvio padrão: 0.0331


## Estabilidade — Jaccard por tópico

Compara keywords do modelo principal (seed=42) com os 4 seeds alternativos. Jaccard < 0.5 indica tópico instável.

In [ ]:
import seaborn as sns
# Jaccard de estabilidade por topico (seeds vs referencia)
import textwrap as _tw_stab

_ref_kws = topics_keywords  # seed principal
_other_seeds = [s for s in results_per_seed if s != seed]
_jac_rows = []
for s in _other_seeds:
    _skws = results_per_seed[s]
    for tid in range(BEST_K):
        _r = set(_ref_kws.get(tid, []))
        _o = set(_skws.get(tid, []))
        _u = _r | _o
        j = len(_r & _o) / len(_u) if _u else 0.0
        _jac_rows.append({"seed": str(s), "topic_id": tid, "jaccard": j})

_jac_df = pd.DataFrame(_jac_rows)
_topic_order = (_jac_df.groupby("topic_id")["jaccard"].mean()
                .sort_values(ascending=True).index.tolist())
_jlabels = {t: f"T{t}: {_tw_stab.shorten(str(names.get(t,"")), 30, placeholder="...")}" for t in range(BEST_K)}
_jac_df["topic_label"] = _jac_df["topic_id"].map(_jlabels)
_ordered = [_jlabels[t] for t in _topic_order]

fig, ax = plt.subplots(figsize=(9, max(5, 0.45 * BEST_K)))
sns.boxplot(data=_jac_df, x="jaccard", y="topic_label", order=_ordered,
            palette="RdYlGn", orient="h", ax=ax)
ax.axvline(x=0.5, color="red", linestyle="--", alpha=0.5, linewidth=1,
           label="limiar 0.5")
ax.set_xlabel("Jaccard (vs seed referencia)", fontsize=10)
ax.set_title(f"Estabilidade por topico (LDA, {len(_other_seeds)} seeds)", fontsize=12, fontweight="bold")
ax.set_xlim(0, 1)
ax.legend(fontsize=9)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "lda_stability_boxplot.png", dpi=150, bbox_inches="tight", facecolor="white")
plt.show()
print(f"Salvo: {OUTPUT_DIR / 'lda_stability_boxplot.png'}")


## 10. Distribuição de documentos por tópico

Mostra balanceamento dos clusters. Distribuição muito desigual pode indicar:
- 1 tópico "lixo" capturando outliers
- K muito alto fragmentando o sinal
- Categorias do corpus naturalmente desbalanceadas (Reuters: `earn` é ~40% dos docs)

In [ ]:
# Cell 18 — Bar chart docs por topico
topic_doc_counts = pd.Series(dominant).value_counts().sort_index()

fig, ax = plt.subplots(figsize=(11, 4.5))
labels = [f"T{tid}\n{names[tid][:22]}" for tid in topic_doc_counts.index]
bars = ax.bar(topic_doc_counts.index, topic_doc_counts.values, color="steelblue")
ax.set_xticks(topic_doc_counts.index); ax.set_xticklabels(labels, fontsize=8, rotation=30, ha="right")
ax.set_ylabel("# documentos dominantes")
ax.set_title(f"Distribuição de docs por tópico (n={len(docs):,})")
for bar, c in zip(bars, topic_doc_counts.values):
    ax.text(bar.get_x() + bar.get_width()/2, c + max(topic_doc_counts)*0.01,
            f"{c:,}\n({c/len(docs)*100:.1f}%)", ha="center", fontsize=8)
plt.tight_layout(); plt.show()

print(f"\nDocs por tópico — descritivas: {topic_doc_counts.describe().to_dict()}")

In [ ]:
# Cell 19 — Heatmap theta (doc x topico) — sample
sample_n = min(200, len(full))
sample_idx = np.random.RandomState(42).choice(len(full), sample_n, replace=False)
theta_sample = np.array([full[i] for i in sample_idx])

# Sort sample by dominant topic for readability
sample_dom = theta_sample.argmax(axis=1)
order = np.argsort(sample_dom)
theta_sorted = theta_sample[order]

fig, ax = plt.subplots(figsize=(12, 5))
sns.heatmap(theta_sorted.T, cmap="YlOrRd", cbar_kws={"label": "P(tópico | doc)"},
            xticklabels=False, yticklabels=[f"T{i}" for i in range(BEST_K)], ax=ax)
ax.set_xlabel(f"Documentos (sample {sample_n}, ordenados por tópico dominante)")
ax.set_ylabel("Tópico"); ax.set_title("Distribuição θ (doc → tópico)")
plt.tight_layout(); plt.show()

## t-SNE do espaço de tópicos (θ)

Reduz a matriz θ (N×K) a 2D via t-SNE. Gera PNG colorido por tópico dominante e versão HTML interativa com hover do texto.

In [ ]:
import plotly.express as px
from sklearn.manifold import TSNE

_theta_mat = np.array(full)  # (n_docs, K)
_texts_lda = [t[:100] + "..." if len(t) > 100 else t for t in docs]

print("Calculando t-SNE sobre matriz theta...")
_tsne = TSNE(n_components=2, perplexity=30, random_state=seed, max_iter=500)
_theta_2d = _tsne.fit_transform(_theta_mat)

_cmap_lda = plt.get_cmap("tab20", max(BEST_K, 1))

# PNG por topico dominante
fig, ax = plt.subplots(figsize=(11, 8))
for tid in range(BEST_K):
    _mm = np.array(dominant) == tid
    if _mm.any():
        import textwrap as _tw_u
        _nm = _tw_u.shorten(str(names.get(tid, "")), 20, placeholder="...")
        ax.scatter(_theta_2d[_mm, 0], _theta_2d[_mm, 1],
                   c=[_cmap_lda(tid)], s=14, alpha=0.72, linewidths=0,
                   label=f"T{tid}: {_nm}")
ax.set_title("t-SNE do espaco theta — topico dominante (LDA)", fontsize=13, fontweight="bold")
ax.set_xlabel("t-SNE dim 1"); ax.set_ylabel("t-SNE dim 2")
ax.legend(loc="best", fontsize=7, framealpha=0.85, ncol=2)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "lda_tsne_theta_topico.png", dpi=150, bbox_inches="tight", facecolor="white")
plt.show()
print(f"Salvo: {OUTPUT_DIR / 'lda_tsne_theta_topico.png'}")

# HTML interativo por topico dominante
import textwrap as _tw_u2
_tsne_df_lda = pd.DataFrame({
    "x": _theta_2d[:, 0], "y": _theta_2d[:, 1],
    "Texto": _texts_lda,
    "Topico": [f"T{t}: {_tw_u2.shorten(str(names.get(t,"")), 25, placeholder="...")}" for t in dominant],
})
fig_u = px.scatter(
    _tsne_df_lda, x="x", y="y", color="Topico",
    hover_data={"x": False, "y": False, "Texto": True},
    title="t-SNE espaco theta — topico dominante (interativo, LDA)",
    labels={"x": "t-SNE dim 1", "y": "t-SNE dim 2"}, opacity=0.72,
)
fig_u.update_traces(marker_size=5)
_out_u = OUTPUT_DIR / "lda_tsne_theta_interactive.html"
fig_u.write_html(_out_u)
fig_u.show()
print(f"Salvo: {_out_u}")


## 11. Cross-tab tópico × categoria Reuters

Reuters tem categorias originais (`earn`, `acq`, `crude`, etc.). Comparamos os tópicos descobertos pelo LDA com as categorias **manuais** — uma forma de validação implícita.

In [ ]:
# Cell 20 — Heatmap topic x category
df_topic = df[["category"]].copy()
df_topic["topic_id"] = dominant
df_topic["topic_name"] = [names[t] for t in dominant]

# Top categorias com volume relevante
top_cats = df["category"].value_counts().head(15).index.tolist()
df_filt = df_topic[df_topic["category"].isin(top_cats)]

ct = pd.crosstab(df_filt["topic_id"], df_filt["category"], normalize="index")
ct = ct[top_cats]  # mantem ordem das top categorias
ct.index = [f"T{i} {names[i][:25]}" for i in ct.index]

fig, ax = plt.subplots(figsize=(13, 0.5*len(ct) + 2))
sns.heatmap(ct, annot=True, fmt=".2f", cmap="YlOrRd", cbar_kws={"label": "P(category | topic)"},
            linewidths=0.3, ax=ax, annot_kws={"fontsize": 8})
ax.set_xlabel("Categoria Reuters (top 15)"); ax.set_ylabel("Tópico LDA")
ax.set_title("Cross-tab: distribuição de categorias Reuters por tópico LDA")
plt.tight_layout(); plt.show()

## 12. pyLDAvis — visualização interativa

Visualização canônica do LDA (Sievert & Shirley 2014). Mostra:
- **Esquerda**: tópicos como círculos no plano 2D (PCA do espaço inter-tópico). Tamanho = prevalência. Distância = dissimilaridade.
- **Direita**: bar chart das top-30 palavras do tópico selecionado, com slider λ controlando "exclusivity vs frequency" (Bischof & Airoldi 2016 — base do FREX score).

In [ ]:
# Cell 21 — pyLDAvis (interativo, salva HTML)
import pyLDAvis
import pyLDAvis.gensim_models as gensim_pyLDAvis

pyLDAvis.enable_notebook()

print("Preparing pyLDAvis (~30s)...")
vis_data = gensim_pyLDAvis.prepare(model, corpus_bow, dictionary, sort_topics=False)
out_html = f"{out_dir}/lda_pyldavis_K{BEST_K}.html"
pyLDAvis.save_html(vis_data, out_html)
print(f"saved: {out_html}")
vis_data

## 13. Top documentos por tópico

Mostra os 3 docs com maior probabilidade em cada tópico — verificação qualitativa direta da semântica.

In [ ]:
# Cell 22 — Top 3 docs por topico
TOP_N_DOCS = 3
TRUNC = 200

theta_arr = np.array(full)
print("Top docs por topico (truncados em 200 chars):\n")
for tid in sorted(topics_keywords):
    top_idx = np.argsort(theta_arr[:, tid])[::-1][:TOP_N_DOCS]
    print(f"=== T{tid} [{names[tid]}] ===")
    print(f"  keywords: {topics_keywords[tid][:8]}\n")
    for rank, idx in enumerate(top_idx, 1):
        prob = theta_arr[idx, tid]
        cat = df.iloc[idx]["category"]
        text = df.iloc[idx]["message"][:TRUNC].replace("\n", " ")
        print(f"  #{rank} (P={prob:.3f}, cat={cat}): {text}...")
        print()

## 14. Export + relatório qualitativo

Gera os CSVs canônicos (`lda_results.csv`, `lda_topics_for_eval.csv`, `lda_metrics.csv`).

In [ ]:
# Cell 23 — Export final
os.makedirs(out_dir, exist_ok=True)

export_results(
    topics=dominant, probs=full, names=names, texts=docs,
    output_path=f"{out_dir}/lda_results.csv",
    topic_type="probabilistic", granularity="unit", post_ids=post_ids,
)
export_topics_for_eval(
    topics_keywords=topics_keywords, topics_names=names,
    model_name="lda", output_path=f"{out_dir}/lda_topics_for_eval.csv",
)
metrics_full = {**metrics, "k_grid_scores": json.dumps(scores),
                "k_perplexity_scores": json.dumps(perplexity_scores), "grid_passes": 20,
                "best_alpha": str(best_alpha), "best_eta": str(best_eta)}
pd.DataFrame([metrics_full]).to_csv(f"{out_dir}/lda_metrics.csv", index=False)
# Cache acessível em próximos runs (foi salvo no run dir; copia pro diretório-base)
pd.DataFrame([metrics_full]).to_csv(f"{BASE_OUTPUT_DIR}/lda_metrics.csv", index=False)

print(f"CSVs em {out_dir}/:")
for f in ["lda_results.csv", "lda_topics_for_eval.csv", "lda_metrics.csv"]:
    sz = os.path.getsize(f"{out_dir}/{f}")
    print(f"  {f} ({sz/1024:.0f} KB)")
print(f"  lda_pyldavis_K{BEST_K}.html")

In [ ]:
# Cell 24 — Relatorio qualitativo (template)
md = qualitative_report(topics_keywords, names)
print(md)

## Próximos passos

- Comparar com STM (`04_stm_reuters.ipynb`) — esperado: macro-tópicos idênticos, validação cross-algorithm
- Aplicar o mesmo template em Folha (`02_lda_folha.ipynb`) — PT-BR, ~165k docs
- NPMI LDA↔STM intra-corpus no notebook comparativo (`05_comparativo.ipynb`)

## Referências

- Apté, C., Damerau, F., Weiss, S. M. (1994). *Automated Learning of Decision Rules for Text Categorization*. ACM TOIS — split ModApte.
- Blei, D. M., Ng, A. Y., Jordan, M. I. (2003). *Latent Dirichlet Allocation*. JMLR.
- Roder, M., Both, A., Hinneburg, A. (2015). *Exploring the Space of Topic Coherence Measures*. WSDM — c_v.
- Stevens, K. et al. (2012). *Exploring topic coherence over many models and many topics*. EMNLP — limitações de c_v com K alto.
- Dieng, A. B., Ruiz, F. J. R., Blei, D. M. (2020). *Topic Modeling in Embedding Spaces*. TACL — Topic Diversity.
- Sievert, C., Shirley, K. (2014). *LDAvis: A method for visualizing and interpreting topics*. ACL Workshop — pyLDAvis.
- Bischof, J. M., Airoldi, E. M. (2016). *Summarizing topical content with word frequency and exclusivity*. ICML — FREX score.
